# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from _import import *

# Beams

In [ ]:
with open("data/X-beam-final_named_structure.txt", "r") as f:
    beam_lines = f.readlines()
    # Skip header column
    beam_lines = beam_lines[1:]
    # Split by tabs, columns: beam, node, x, y, z
    beam_lines = [line.split("\t") for line in beam_lines]
    # Skip nodes with empty coordinates
    beam_lines = [
        line for line in beam_lines if all(coord.strip() != "" for coord in line[2:5])
    ]
    # Make data frame based on node numbers
    beam_lines = pd.DataFrame(
        beam_lines, columns=["Beam", "Node", "X", "Y", "Z"]
    ).astype({"Beam": str, "Node": int, "X": float, "Y": float, "Z": float})
beam_lines

In [ ]:
beams = pd.read_csv("all_beam_nodes_annot.csv")
node_to_beam = beams.groupby("Node").first()["Beam"].to_dict()

In [ ]:
plot_bridge_3d_structure(
    highlight_nodes=beams["Node"].unique().tolist(),
    annotations=node_to_beam
)

In [ ]:
diff = set(I_BEAM_NODES).symmetric_difference(set(NODES_MISSING_STRESS))
set(diff) == set(NODES_MISSING_COORDS)

In [ ]:
df = read_data_file(*(0,0,0,0,0))
df[df['Node Number'].isin(I_BEAM_NODES)].head()

In [ ]:
plot_bridge_3d_variable_over_time_df(
    df,
    var_name="TotalDeformation",
)

# Look at delta nodes over variables

In [ ]:
df = get_data_variable_and_region_aggregated((1,0,0))

In [ ]:
df

In [ ]:
delta_df = filter_df_to_delta_nodes(
    df, 
    variable_names=["DirectionalDeformation_Z_axis"], 
    top_pct=500, 
    aggregate_by_time=True  
)[0]


In [ ]:
delta_nodes = delta_df[(delta_df["delta_health"]==0)&(delta_df["scenario"]==32)]["Node Number"].unique().tolist()
plot_bridge_3d_structure(
    highlight_nodes=delta_nodes,
)

In [ ]:
plot_bridge_3d_variable_over_time_df(
    delta_df[delta_df["delta_health"]==0],
    var_name="DirectionalDeformation_Z_axis",
)

# Nearest Neighbours

In [ ]:
import pickle
import numpy as np
from sklearn.neighbors import NearestNeighbors

def write_nearest_neighbours_file(df: pd.DataFrame, N_neighbours: int):
    # 1. Get unique Node Coordinates
    # We assume 'df' contains all nodes. We drop duplicates to get one row per node.
    unique_nodes = df[['Node Number', 'X', 'Y', 'Z']].drop_duplicates('Node Number').set_index('Node Number')

    # 2. Fit Nearest Neighbors
    # k=6 because the closest neighbor is the node itself (distance 0).
    # We want the 5 closest *other* nodes.
    nbrs = NearestNeighbors(n_neighbors=N_neighbours+1, algorithm='ball_tree').fit(unique_nodes[['X', 'Y', 'Z']])
    distances, indices = nbrs.kneighbors(unique_nodes[['X', 'Y', 'Z']])

    # 3. Create a dictionary: Node_ID -> [List of 5 Neighbor Node_IDs]
    node_ids_array = unique_nodes.index.to_numpy()
    neighbor_map = {}

    for i, row_indices in enumerate(indices):
        current_node = node_ids_array[i]
        # row_indices[0] is the node itself. Slice [1:] to get the 5 real neighbors.
        neighbor_indices = row_indices[1:]
        neighbor_ids = node_ids_array[neighbor_indices]
        neighbor_map[current_node] = neighbor_ids

    # 4. Save to disk
    filename = f'node_neighbors_{N_neighbours}.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(neighbor_map, f)

    print(f"Successfully computed {N_neighbours} nearest neighbors for {len(neighbor_map)} nodes.")
    print(f"Saved mapping to {filename}")

In [ ]:
# Params
N_neighbours = 20
target_node = node_id  # use existing variable

# Coordinates table
coords_df = (
    beam_lines[["Node", "X", "Y", "Z"]]
    .drop_duplicates("Node")
    .set_index("Node")
)

if target_node not in coords_df.index:
    raise ValueError(f"Node {target_node} not found in beam_lines.")

# Fit NN
nbrs = NearestNeighbors(n_neighbors=N_neighbours + 1, algorithm="ball_tree").fit(
    coords_df[["X", "Y", "Z"]]
)
distances, indices = nbrs.kneighbors(
    coords_df.loc[[target_node], ["X", "Y", "Z"]]
)

# Neighbour ids (exclude self)
neighbor_idx = indices[0][1:]
neighbor_ids = coords_df.index.to_numpy()[neighbor_idx]
neighbor_coords = coords_df.loc[neighbor_ids, ["X", "Y", "Z"]].to_numpy()

# Sphere radius = farthest neighbour distance
radius = distances[0][1:].max()

# Sphere mesh
u = np.linspace(0, 2 * np.pi, 40)
v = np.linspace(0, np.pi, 20)
uu, vv = np.meshgrid(u, v)
cx, cy, cz = coords_df.loc[target_node, ["X", "Y", "Z"]].to_numpy()
xs = cx + radius * np.cos(uu) * np.sin(vv)
ys = cy + radius * np.sin(uu) * np.sin(vv)
zs = cz + radius * np.cos(vv)

# Plot
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")

# neighbours + target
ax.scatter(neighbor_coords[:, 0], neighbor_coords[:, 1], neighbor_coords[:, 2],
           s=15, alpha=0.8, label="Nearest neighbours")
ax.scatter([cx], [cy], [cz], s=60, color="red", label="Target node")

# sphere
ax.plot_surface(xs, ys, zs, alpha=0.15, color="gray", linewidth=0)

ax.set_title(f"Node {target_node} + {N_neighbours} nearest neighbours")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend(loc="best")
plt.tight_layout()

In [ ]:
df = get_data_variable_aggregated((0,0,0,6),filter_out_invalid_nodes=True)
df = add_graphic_delta_nodes(df)
df[df["delta_health"]==0]

In [ ]:
import warnings

# Silence the groupby.apply future warning
warnings.filterwarnings(
    "ignore",
    message="DataFrameGroupBy.apply operated on the grouping columns",
    category=FutureWarning,
)

# Patch Series.corr to avoid warnings on constant inputs
if not hasattr(pd.Series, "_orig_corr"):
    pd.Series._orig_corr = pd.Series.corr

def _safe_corr(self, other, method="pearson", min_periods=None):
    a = self
    b = other
    mask = a.notna() & b.notna()
    a = a[mask]
    b = b[mask]
    if a.nunique(dropna=True) < 2 or b.nunique(dropna=True) < 2:
        return float("nan")
    return pd.Series._orig_corr(a, b, method=method, min_periods=min_periods)

pd.Series.corr = _safe_corr

In [ ]:
import numpy as np
import pandas as pd
import pickle
from collections import OrderedDict

# ---- knobs ----
N_list = [5, 10, 20, 50, 100]
value_col   = "DirectionalDeformation_Z_axis"
label_col   = "delta_health"          # or "delta_health" if that's what you truly want
scenario_col = "scenario"
time_col     = "time"
node_col     = "Node Number"
corr_value_col = "Z_resid"

# how to compress time series -> scalar per (scenario,node)
AGG = "mean_abs"  # options: "max_abs", "mean_abs", "rms"

def agg_residual(x: pd.Series, mode: str) -> float:
    a = x.to_numpy(dtype=float)
    if mode == "max_abs":
        return np.nanmax(np.abs(a))
    elif mode == "mean_abs":
        return np.nanmean(np.abs(a))
    elif mode == "rms":
        return float(np.sqrt(np.nanmean(a * a)))
    else:
        raise ValueError(f"Unknown AGG={mode}")

summary_records = []

# health label table: one row per (scenario,node)
health_tbl = (
    df[[scenario_col, node_col, label_col]]
    .drop_duplicates(subset=[scenario_col, node_col])
    .copy()
)

for N_neighbours in N_list:
    df_temp = df.copy()

    # 1) ensure / write mapping (your function)
    write_nearest_neighbours_file(df_temp, N_neighbours=N_neighbours)

    # 2) load mapping
    with open(f"node_neighbors_{N_neighbours}.pkl", "rb") as f:
        neighbor_map = pickle.load(f)

    print(f"\nN_neighbours={N_neighbours}: computing residuals...")

    # 3) wide: rows=(scenario,time), cols=node_id
    df_wide = df_temp.pivot_table(
        index=[scenario_col, time_col],
        columns=node_col,
        values=value_col,
        aggfunc="mean",
    )

    # 4) compute neighbor means (wide)
    local_mean_wide = pd.DataFrame(index=df_wide.index, columns=df_wide.columns, dtype=float)

    cols_set = set(df_wide.columns)

    for node_id in df_wide.columns:
        neigh = neighbor_map.get(node_id, [])
        # enforce exactly N neighbours and exclude self if present
        neigh = [nid for nid in neigh if nid != node_id]
        neigh = neigh[:N_neighbours]
        valid = [nid for nid in neigh if nid in cols_set]
        if valid:
            local_mean_wide[node_id] = df_wide[valid].mean(axis=1, skipna=True)

    # 5) residual in wide (no merge-back needed)
    resid_wide = df_wide - local_mean_wide

    # 6) long residuals (keep only valid residuals)
    resid_long = (
        resid_wide.stack(dropna=True)
        .rename(f"{corr_value_col}")
        .reset_index()
        .rename(columns={"level_2": node_col})
    )

    # 7) attach health (one label per scenario-node)
    resid_long = resid_long.merge(
        health_tbl,
        on=[scenario_col, node_col],
        how="left",
        validate="many_to_one",
    )

    # drop rows without label (if any)
    resid_long = resid_long.dropna(subset=[label_col])

    # 8) aggregate over time -> one row per (scenario,node)
    agg_tbl = (
        resid_long
        .groupby([scenario_col, node_col], as_index=False)
        .agg(
            Z_resid_agg=(f"{corr_value_col}", lambda s: agg_residual(s, AGG)),
            label=(label_col, "first"),
            n_time=(f"{corr_value_col}", "count"),
        )
    )

    # coverage stats
    coverage = agg_tbl["n_time"].describe()

    # 9) correlations on aggregated table (the meaningful one)
    pearson_all  = agg_tbl["Z_resid_agg"].corr(agg_tbl["label"], method="pearson")
    spearman_all = agg_tbl["Z_resid_agg"].corr(agg_tbl["label"], method="spearman")

    # 10) group stats by label on aggregated table
    group_stats = (
        agg_tbl.groupby("label")["Z_resid_agg"]
        .agg(["mean", "median", "count"])
        .to_dict()
    )

    # 11) per-node correlation across scenarios (only nodes whose label varies across scenarios)
    per_node_corr = (
        agg_tbl.groupby(node_col)
        .apply(lambda g: g["Z_resid_agg"].corr(g["label"]) if g["label"].nunique() > 1 else np.nan)
        .dropna()
    )

    per_node_summary = OrderedDict()
    top_nodes, top_corrs = [], []
    if not per_node_corr.empty:
        per_node_summary["per_node_count"] = int(per_node_corr.shape[0])
        per_node_summary["per_node_mean_corr"] = float(per_node_corr.mean())
        per_node_summary["per_node_median_corr"] = float(per_node_corr.median())
        per_node_summary["per_node_std_corr"] = float(per_node_corr.std())
        per_node_summary["per_node_25pct"] = float(per_node_corr.quantile(0.25))
        per_node_summary["per_node_75pct"] = float(per_node_corr.quantile(0.75))

        top_abs = per_node_corr.abs().sort_values(ascending=False).head(10)
        top_nodes = list(top_abs.index)
        top_corrs = [float(per_node_corr.loc[n]) for n in top_nodes]
    else:
        per_node_summary["per_node_count"] = 0

    # 12) summary record
    summary_records.append({
        "N_neighbours": N_neighbours,
        "agg_mode": AGG,
        "pearson_all": pearson_all,
        "spearman_all": spearman_all,
        "mean_resid_label_0": group_stats["mean"].get(0, np.nan) if "mean" in group_stats else np.nan,
        "mean_resid_label_1": group_stats["mean"].get(1, np.nan) if "mean" in group_stats else np.nan,
        "count_label_0": group_stats["count"].get(0, 0) if "count" in group_stats else 0,
        "count_label_1": group_stats["count"].get(1, 0) if "count" in group_stats else 0,
        "median_n_time": float(coverage["50%"]),
        "min_n_time": float(coverage["min"]),
        "max_n_time": float(coverage["max"]),
        **per_node_summary,
        "top_nodes_by_abs_corr": top_nodes,
        "top_node_corrs": top_corrs,
    })

    # concise report
    print(f"AGG={AGG} | Pearson={pearson_all:.4f} Spearman={spearman_all:.4f}")
    print(f"label=0 mean={summary_records[-1]['mean_resid_label_0']:.6e} (n={summary_records[-1]['count_label_0']})")
    print(f"label=1 mean={summary_records[-1]['mean_resid_label_1']:.6e} (n={summary_records[-1]['count_label_1']})")
    print(f"coverage n_time: median={summary_records[-1]['median_n_time']:.1f}, min={summary_records[-1]['min_n_time']:.0f}, max={summary_records[-1]['max_n_time']:.0f}")
    if per_node_summary["per_node_count"] > 0:
        print(f"Per-node corr (across scenarios) median={per_node_summary['per_node_median_corr']:.4f}")
        print("Top nodes by |corr|:")
        for nid, c in zip(top_nodes, top_corrs):
            print(f"  {nid}: {c:.4f}")
    else:
        print("Per-node correlations empty (labels constant per node across scenarios).")

# final summary df
summary_df = pd.DataFrame(summary_records).set_index("N_neighbours").sort_index()
summary_df


In [ ]:
# filter outliers
resid_long_plot = resid_long[resid_long["Z_resid"].abs() < resid_long["Z_resid"].abs().quantile(0.95)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

colors = {0: "tab:red", 1: "tab:blue"}
for ax_i, dh in zip(axes, [0, 1]):
    sub = resid_long_plot[resid_long_plot["delta_health"] == dh]
    sub.plot.scatter(
        x="time",
        y="Z_resid",
        c=colors[dh],
        alpha=0.5,
        s=10,
        ax=ax_i,
    )
    ax_i.set_title(f"Z_resid (delta_health={dh})")
    ax_i.set_xlabel("time")
    ax_i.set_ylabel("Z_resid")

plt.tight_layout()

# Visualize defective regions

In [ ]:
beam_df = pd.read_csv("all_beam_nodes_annot.csv")
defective_nodes = beam_df[beam_df["Beam"].isin(np.concatenate(list(REGION_BEAMS.values())))]["Node"].unique().tolist()
plot_bridge_3d_structure(defective_nodes)

In [ ]:
plot_bridge_3d_structure(DEFECTIVE_NODES_BY_REGION[6])

Find variable with highest intersection with defective nodes

In [ ]:
results = []
threshs = [50, 100, 200, 300, 400, 500, 600]
for combo in combinations_region_agg:
    df = get_data_variable_and_region_aggregated((1,0,0))
    combo_key = f"{combo}"
    for var in VARIABLE_NAMES:
        for thresh in threshs:
            delta_df = filter_df_to_delta_nodes(
                df,
                variable_names=[var],
                top_pct=thresh,
                aggregate_by_time=True,
            )[0]
            delta_nodes = delta_df[delta_df["delta_health"] == 0]["Node Number"].unique().tolist()
            intersection = set(defective_nodes).intersection(delta_nodes)
            union = set(defective_nodes).union(delta_nodes)
            results.append(
                {
                    "combo": combo_key,
                    "variable": var,
                    "top_pct": thresh,
                    "intersection_size": len(intersection),
                    "IOU": len(intersection) / len(union),
                }
            )

            if var==VARIABLE_NAMES[-1]:
                delta_df = filter_df_to_delta_nodes(
                df,
                variable_names=["DirectionalDeformation_X_axis","DirectionalDeformation_Z_axis"],
                top_pct=thresh,
                aggregate_by_time=True,
            )[0]
                var = "Combo_XZ"
                delta_nodes = delta_df[delta_df["delta_health"] == 0]["Node Number"].unique().tolist()
                intersection = set(defective_nodes).intersection(delta_nodes)
                union = set(defective_nodes).union(delta_nodes)
                results.append(
                    {
                        "combo": combo_key,
                        "variable": var,
                        "top_pct": thresh,
                        "intersection_size": len(intersection),
                        "IOU": len(intersection) / len(union),
                    }
                )

In [ ]:

results_df = pd.DataFrame(results)
results_df.to_csv("defective_intersections_by_combo.csv", index=False)

avg_df = (
    results_df.groupby(["variable", "top_pct"])["IOU"]
    .mean()
    .reset_index()
)
ax = avg_df.pivot(index="top_pct", columns="variable", values="IOU").plot(
    marker="o", figsize=(8, 5)
)
ax.set_title("Intersection over Union by Variable")
ax.set_xlabel("Delta nodes threshold")
ax.set_ylabel("Intersection over Union")
ax.legend(title="variable", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig("visualization/defective_intersection_by_variable.svg")
plt.show()

# Test data

# Plotting

In [ ]:
plot_nodes_time_series_for_combinations(
    combinations=[(3, 0, 0, 5, 3), (3, 0, 0, 0, 3)],
    node_numbers=[100], 
    variable="DirectionalDeformation_Z_axis"
)


In [ ]:
df = read_data_file(*(0, 0, 0, 0, 3))

In [ ]:
plot_nodes_time_series(df, 1882, variable="DirectionalDeformation_Z_axis")

In [ ]:
unequal = []
for node_number in df["Node Number"].unique():
    fft_90 = compute_node_fft(df, node_number=node_number, scenario_id=90, variable="DirectionalDeformation_X_axis")
    fft_84 = compute_node_fft(df, node_number=node_number, scenario_id=84, variable="DirectionalDeformation_X_axis")
    if not np.array_equal(fft_90, fft_84):
        print(node_number)
        print(f"Difference f: {np.abs(fft_90[0] - fft_84[0]).max()}")
        print(f"Difference m: {np.abs(fft_90[1] - fft_84[1]).max()}")
        unequal.append(node_number)

In [ ]:
compare_bridge_dynamics(df, 24356, 84, 90, variable="EquivalentStress")

In [ ]:
combo = (3, 0, 0, 5, 0)
var_name = VARIABLE_NAMES[combo[-1]]
diff = get_variable_difference_between_combinations(
    (*combo[:3], 0, combo[-1]),
    combo,
    top_pct=5
)
plot_bridge_3d_variable_over_time_df(diff, var_name)

# Voxel

In [ ]:
vx = create_multichannel_voxel_dataset(df)
vx[0].shape

In [ ]:
visualize_voxel_snapshot(vx[0])

# Check for missing nodes (raw data)

In [ ]:
# For raw df:

# Check which nodes have no data for the raw stress and deformation variables
df_stress = read_data_file(0, 0, 0, 0, 7, filter_out_invalid_nodes=False)
df_deformation = read_data_file(0, 0, 0, 0, 0, filter_out_invalid_nodes=False)
# Get nodes missing loads for stress and deformation
def check_missing_loads(df_loads, variable_name):
    missing_loads_nodes = df_loads[df_loads.isnull().any(axis=1)]["Node Number"].unique()
    if len(missing_loads_nodes) > 0:
        print(f"Warning: {len(missing_loads_nodes)} nodes are missing loads for {variable_name}.")
    return missing_loads_nodes
missing_loads_deformation = check_missing_loads(df_deformation, "deformation")
missing_loads_stress = check_missing_loads(df_stress, "stress")
# Get node locations of missing stress
# missing_stress_nodes = COORDS_DF[COORDS_DF["Node Number"].isin(missing_loads_stress)]["Node Number"].unique()
# missing_stress_nodes


In [ ]:
# Compare contents regardless of order/duplicates
set_a = set(np.asarray(missing_loads_deformation).tolist())
set_b = set(NODES_MISSING_DEFORMATION)

print("Equal sets:", set_a == set_b)
print(f"len(missing_loads_deformation) = {len(missing_loads_deformation)}")
print(f"len(NODES_MISSING_DEFORMATION) = {len(NODES_MISSING_DEFORMATION)}")
print(f"len(set(missing_loads_deformation)) = {len(set_a)}")
print(f"len(set(NODES_MISSING_DEFORMATION)) = {len(set_b)}")

only_in_a = sorted(set_a - set_b)
only_in_b = sorted(set_b - set_a)
print("Only in missing_loads_deformation (sample):", only_in_a[:20])
print("Only in NODES_MISSING_DEFORMATION (sample):", only_in_b[:20])
# Optionally assert equality
# assert set_a == set_b, "The two collections do not contain the same elements."

In [ ]:
# Compare contents regardless of order/duplicates
set_a = set(np.asarray(missing_loads_stress).tolist())
set_b = set(NODES_MISSING_STRESS)

print("Equal sets:", set_a == set_b)
print(f"len(missing_loads_stress) = {len(missing_loads_stress)}")
print(f"len(NODES_MISSING_STRESS) = {len(NODES_MISSING_STRESS)}")
print(f"len(set(missing_loads_stress)) = {len(set_a)}")
print(f"len(set(NODES_MISSING_STRESS)) = {len(set_b)}")

only_in_a = sorted(set_a - set_b)
only_in_b = sorted(set_b - set_a)
print("Only in missing_loads_stress (sample):", only_in_a[:20])
print("Only in NODES_MISSING_STRESS (sample):", only_in_b[:20])
# Optionally assert equality
# assert set_a == set_b, "The two collections do not contain the same elements."

Check nodes that have deformation but no stress

In [ ]:
deformation_no_stress = set(NODES_MISSING_STRESS) - set(NODES_MISSING_DEFORMATION)

vals0 = np.array([4.1982e-02, 1.1874e+00, 3.9019e+00, 5.8475e+00, 1.0916e+01,
       1.7553e+01, 2.1460e+01, 3.0450e+01, 4.1010e+01, 4.3895e+01])
vals1 = np.array([4.1982e-02, 1.1874e+00, 3.9019e+00, 3.9018e+00, 5.8475e+00, 5.8476e+00,
 5.8474e+00, 1.0916e+01, 1.0915e+01, 1.7553e+01, 2.1460e+01, 3.0450e+01,
 3.0451e+01, 4.1010e+01, 4.3895e+01, 4.3896e+01])
check_vals = [vals0, vals1]
unique_vals = []
for combo in combinations:
    if combo[-1] == 0:
        df_deformation = read_data_file(*combo)
        vals = df_deformation[df_deformation["Node Number"].isin(deformation_no_stress)]["TotalDeformation"].unique()
        # print(vals)
        # if not np.all(np.isclose(vals, check_vals[combo[0]])):
        #     break
        unique_vals.append((vals,(combo,combo[-2])))

In [ ]:
from collections import defaultdict

unique_python_vals = []
array_to_regions = defaultdict(set)

for arr, combo in unique_vals:
    found = False
    for i, u in enumerate(unique_python_vals):
        if np.array_equal(arr, u):
            array_to_regions[i].add(combo)
            found = True
            break
    if not found:
        unique_python_vals.append(arr)
        array_to_regions[len(unique_python_vals) - 1].add(combo)

# Show arrays with all the regions that have them
for i, arr in enumerate(unique_python_vals):
    print(f"Array {i}: {arr}, Regions: {sorted(array_to_regions[i])}")
unique_python_vals

In [ ]:
plot_bridge_3d_structure(highlight_nodes=missing_stress_nodes, s=3)

Save to file

In [ ]:

# Save nodes without stress
with open("nodes_missing_stress.txt", "w") as f:
    for node in sorted(missing_loads_stress):
        f.write(f"{node}\n")
# Save nodes without deformation (and no coordinates)
with open("nodes_missing_deformation.txt", "w") as f:
    for node in sorted(missing_loads_deformation):
        f.write(f"{node}\n")
# Node numbers to be used have both coordinates and loads
valid_node_numbers = set(NODE_NUMBERS) - set(missing_loads_deformation).union(set(missing_loads_stress))
assert len(valid_node_numbers) == len(NODE_NUMBERS) - len(set(missing_loads_deformation).union(set(missing_loads_stress)))
print(f"{len(valid_node_numbers)} Node numbers have both coordinates and loads.")
with open("valid_node_numbers.txt", "w") as f:
    for node in sorted(valid_node_numbers):
        f.write(f"{node}\n")

# Check for missing nodes (variable aggregated df)

In [ ]:
df = get_data_variable_aggregated(
    (0, 0, 0, 0), filter_out_invalid_nodes=False
)

In [ ]:
# For aggregated df:

# Check which node numbers have missing coordinates
coords = df["X"]
missing_coords_nodes = df[coords.isna()]["Node Number"].unique()

# Check that no coords nodes are the same as no deformation nodes
assert set(missing_coords_nodes) == set(missing_loads_deformation), "Mismatch between missing coords and deformation loads nodes"
assert set(missing_coords_nodes).issubset(set(missing_loads_stress)), "Some missing coords nodes are not in missing stress loads nodes"

# Check which nodes have missing loads
loads = df[VARIABLE_NAMES]
missing_loads_nodes = df[loads.isna().any(axis=1)]["Node Number"].unique()

print(f"{len(missing_coords_nodes)} Node numbers with missing coordinates:", missing_coords_nodes)
print(f"{len(missing_loads_nodes)} Node numbers with missing loads:", missing_loads_nodes)

loads_no_coords = set(missing_loads_nodes) - set(missing_coords_nodes)
coords_no_loads = set(missing_coords_nodes) - set(missing_loads_nodes)
print(f"{len(loads_no_coords)} Node numbers with missing loads but have coordinates:", loads_no_coords)
print(f"{len(coords_no_loads)} Node numbers with missing coordinates but have loads:", coords_no_loads)

In [ ]:
# Check which sequential node numbers are missing
all_node_numbers = set(range(df["Node Number"].min(), df["Node Number"].max() + 1))
present_node_numbers = set(df["Node Number"].unique())
missing_node_numbers = all_node_numbers - present_node_numbers
print("Missing node numbers:", missing_node_numbers)
# -> None are missing!

In [ ]:
# Save nodes without coordinates
with open("nodes_missing_coords.txt", "w") as f:
    for node in sorted(missing_coords_nodes):
        f.write(f"{node}\n")

# Check for outliers

In [ ]:
outliers = []
for combo in combinations:
    if combo[-1] != 0:
        continue
    df = read_data_file(*combo)
    outliers.append((filter_outliers(df)[1],combo[-2]))

In [ ]:
import pandas as pd
from itertools import combinations

# Analyze correlation between outlier nodes and region


# Prepare a DataFrame from the outliers list
outlier_records = []
for nodes, region in outliers:
    for node in nodes:
        outlier_records.append({'Node Number': int(node), 'region': region})

outlier_df = pd.DataFrame(outlier_records)

region_node_sets = outlier_df.groupby('region')['Node Number'].apply(set)
# Compute intersections between unique outlier nodes of each region

print("Intersections between regions (node counts):")
for r1, r2 in combinations(region_node_sets.index, 2):
    intersection = region_node_sets[r1] & region_node_sets[r2]
    print(f"Regions {r1} & {r2}: {len(intersection)} nodes")
    if intersection:
        print(sorted(intersection))
    print("-" * 40)

for region, nodes in region_node_sets.items():
    print(f"Region {region}: {len(nodes)} unique outlier nodes")
    print(sorted(nodes))
    print("-" * 40)


# Count outlier nodes per region
region_outlier_counts = outlier_df.groupby('region')['Node Number'].nunique()

print("Outlier node counts per region:")
print(region_outlier_counts)

# If you want to visualize the distribution:
import matplotlib.pyplot as plt

region_outlier_counts.plot(kind='bar')
plt.xlabel('Region')
plt.ylabel('Number of Outlier Nodes')
plt.title('Outlier Nodes per Region')
plt.tight_layout()
plt.show()


# 1. Bar plot: Unique outlier nodes per region
plt.figure(figsize=(8, 4))
region_outlier_counts.plot(kind='bar', color='skyblue')
plt.xlabel('Region')
plt.ylabel('Number of Unique Outlier Nodes')
plt.title('Unique Outlier Nodes per Region')
plt.tight_layout()
plt.show()

# 2. Heatmap: Shared outlier nodes between regions
regions = sorted(region_node_sets.index)
intersection_matrix = np.zeros((len(regions), len(regions)), dtype=int)

for i, r1 in enumerate(regions):
    for j, r2 in enumerate(regions):
        if i == j:
            intersection_matrix[i, j] = len(region_node_sets[r1])
        else:
            intersection_matrix[i, j] = len(region_node_sets[r1] & region_node_sets[r2])

plt.figure(figsize=(8, 6))
sns.heatmap(intersection_matrix, annot=True, fmt="d", cmap="YlGnBu",
            xticklabels=regions, yticklabels=regions)
plt.title('Shared Outlier Nodes Between Regions\n(Diagonal = Unique per Region)')
plt.xlabel('Region')
plt.ylabel('Region')
plt.tight_layout()
plt.show()

Show superset of outliers on bridge

In [ ]:
superset_outliers = set()
for nodes, region in outliers:
    superset_outliers.update(nodes)
plot_bridge_3d_structure(highlight_nodes=sorted(superset_outliers), s=3)

In [ ]:
with open("superset_outliers.txt", "w") as f:
    for node in sorted(superset_outliers):
        f.write(f"{node}\n")

# Getting top_pct

In [ ]:
top_pct = [0.05, 0.01, 0.001, 0.0001]


avg_offs = []
num_unique_nodes = []
for i, pct in enumerate(top_pct):
    delta_nodes, diffs = get_delta_nodes(top_pct=pct)
    
    # delta_nodes is a list of 6 arrays, each containing node numbers for one Damage state
    n = len(delta_nodes)
    intersection_matrix = np.zeros((n, n), dtype=int)

    # Aggregate delta nodes across (train_config, load, season) for each variable,
    # then compute & plot intersection matrices (6 damage states) per variable.
    intersections_by_var = {}
    for vidx, vname in enumerate(VARIABLE_NAMES):
        # collect union of nodes for each damage state (1..6) across all scenarios
        damage_sets = []
        for dmg in range(1, 7):
            s = set()
            for key, arr in delta_nodes.items():
                # key format: (train_config, load, season, damage, variable)
                if isinstance(key, tuple) and len(key) == 5:
                    if key[3] == dmg and key[4] == vidx:
                        s.update(np.asarray(arr).tolist())
            damage_sets.append(s)

        # Get set of unique delta nodes 
        num_unique = len(set().union(*damage_sets))
        num_unique_nodes.append((vname, num_unique))

        # compute pairwise intersections
        m = np.zeros((6, 6), dtype=int)
        for i in range(6):
            for j in range(6):
                m[i, j] = len(damage_sets[i] & damage_sets[j])

        intersections_by_var[vname] = m


    # Plot heatmaps for each variable
    import matplotlib.pyplot as plt
    for vname, m in intersections_by_var.items():
        diag = np.diag(m).astype(float)
        denom = diag.copy()
        denom[denom == 0] = 1  # avoid division by zero
        pct = (m.astype(float) / denom[:, None]) * 100

        # average of unique off-diagonal pairwise intersections
        i_upper = np.triu_indices_from(pct, k=1)
        off_vals = pct[i_upper]
        avg_off = off_vals.mean() if off_vals.size else 0.0
        avg_offs.append(avg_off)
        annot = f"Avg non-diagonal pairwise intersection = {avg_off:.2f} (n_pairs={off_vals.size})"

        # plot here with annotation and skip the original plotting below
        mask = np.triu(np.ones_like(pct, dtype=bool), k=1)
        plt.figure(figsize=(5, 4))
        sns.heatmap(
            pct,
            annot=True,
            fmt=".1f",
            cmap="Blues",
            cbar=True,
            mask=mask,
            xticklabels=[f"Damage {i+1}" for i in range(6)],
            yticklabels=[f"Damage {i+1}" for i in range(6)],
        )
        plt.title(f"Intersection of Delta Nodes (%) - {vname}")
        plt.gca().text(0.5, 1.2, annot, ha='center', va='center', transform=plt.gca().transAxes)
        plt.ylabel("Damage State")
        plt.xlabel("Damage State")
        plt.tight_layout()
        plt.show()


# # ensure the rest of the cell (which expects intersection_matrix) uses a sensible default:
# if 'var_name' in globals() and var_name in intersections_by_var:
#     intersection_matrix = intersections_by_var[var_name]
# else:
#     intersection_matrix = list(intersections_by_var.values())[0]
# n = intersection_matrix.shape[0]

# intersection_df = pd.DataFrame(
#     intersection_matrix,
#     index=[f"Damage {i+1}" for i in range(n)],
#     columns=[f"Damage {i+1}" for i in range(n)]
# )

# plt.figure(figsize=(5, 4))
# # Convert absolute intersection counts to percentages (relative to row, i.e., Damage i)
# intersection_pct = intersection_df.div(intersection_df.values.diagonal(), axis=0) * 100
# # Visualize only the lower right triangle (including diagonal)
# mask = np.triu(np.ones_like(intersection_pct, dtype=bool), k=1)
# sns.heatmap(intersection_pct, annot=True, fmt=".1f", cmap="Blues", cbar=True, mask=mask)
# plt.title("Intersection of Delta Nodes Across Damage States (%)")
# plt.ylabel("Damage State")
# plt.xlabel("Damage State")
# plt.tight_layout()
# plt.show()

In [ ]:
# reshape avg offs
num_unique_nodes_array = [num_unique_nodes[i:i+8] for i in range(0, len(num_unique_nodes), 8)]
avg_offs_array = np.array(avg_offs).reshape(len(top_pct), len(VARIABLE_NAMES))

In [ ]:
import pandas as pd
import numpy as np

# Plot average intersection per top_pct for each variable and the overall average across variables.
# Assumes `avg_offs_array`, `top_pct`, and `VARIABLE_NAMES` are already available in the notebook.

import matplotlib.pyplot as plt

# Build DataFrame: rows = top_pct, cols = variables
df_avg = pd.DataFrame(avg_offs_array, index=top_pct, columns=VARIABLE_NAMES)

# Sort by top_pct for a clean log-x plot
df_avg = df_avg.sort_index()

x = np.array(df_avg.index)

plt.figure(figsize=(10, 6))
# plot each variable
for col in df_avg.columns:
    plt.plot(x, df_avg[col].values, marker='o', linestyle='-', alpha=0.6, label=col)

# plot average across variables per top_pct
mean_over_vars = df_avg.mean(axis=1)
plt.plot(x, mean_over_vars.values, marker='s', linestyle='-', color='k', lw=3, label='Average across variables')

plt.xscale('log')
plt.xlabel('Percentage of Top Nodes (top_pct)')
plt.ylabel('Avg non-diagonal pairwise intersection (%)')
plt.title('Average intersection vs top_pct (per variable) and overall average')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, which='both', linestyle='--', alpha=0.3)

# compute average number of unique delta nodes per top_pct (uses num_unique_nodes_array and top_pct)
orig_top = list(top_pct)  # original ordering used when building num_unique_nodes_array
mean_unique_per_pct = [np.mean([t[1] for t in row]) for row in num_unique_nodes_array]

# align to the sorted df_avg.index (x) which may be in different order and contain duplicates
used = [False] * len(orig_top)
mean_unique_aligned = []
for xv in x:
    for i, val in enumerate(orig_top):
        if not used[i] and np.isclose(val, xv):
            mean_unique_aligned.append(mean_unique_per_pct[i])
            used[i] = True
            break
# make a separate plot for avg unique delta nodes per top_pct
plt.figure(figsize=(8, 4))
plt.plot(x, mean_unique_aligned, marker='o', linestyle='--', color='tab:purple', lw=2)
plt.xscale('log')
plt.xlabel('Percentage of Top Nodes (top_pct)')
plt.ylabel('Avg unique delta nodes (count)')
plt.title('Average unique delta nodes (over variables) vs top_pct')
plt.grid(True, which='both', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# keep mean_unique_aligned as a numpy array for any subsequent plotting
mean_unique_aligned = np.array(mean_unique_aligned)
